# Dependencies

In [3]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
# from lead_lag import lag
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from scipy.signal import medfilt
import sys
import logging
from sklearn.preprocessing import StandardScaler # Added for feature scaling


In [5]:

# Configure basic logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# --- 0. Data Loading and Initial Setup ---

def load_your_spot_data(filepath):
    logger.info(f"Loading spot data from {filepath}...")
    try:
        df = pd.read_csv(filepath)
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        df = df.set_index('timestamp').sort_index()

        required_cols = ['best_bid', 'best_ask', 'trade_price', 'trade_quantity']
        for col in required_cols:
            if col not in df.columns:
                raise ValueError(f"Spot data missing required column: '{col}'")

        logger.info(f"Spot data loaded. Shape: {df.shape}")
        return df
    except FileNotFoundError:
        logger.error(f"Spot data file not found: {filepath}")
        sys.exit(1)
    except Exception as e:
        logger.error(f"Error loading spot data: {e}")
        sys.exit(1)

def load_your_perpetual_data(filepath):
    logger.info(f"Loading perpetual data from {filepath}...")
    try:
        df = pd.read_csv(filepath)
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        df = df.set_index('timestamp').sort_index()

        required_cols = ['best_bid', 'best_ask']
        for col in required_cols:
            if col not in df.columns:
                raise ValueError(f"Perpetual data missing required column: '{col}'")

        logger.info(f"Perpetual data loaded. Shape: {df.shape}")
        return df
    except FileNotFoundError:
        logger.error(f"Perpetual data file not found: {filepath}")
        sys.exit(1)
    except Exception as e:
        logger.error(f"Error loading perpetual data: {e}")
        sys.exit(1)

# --- Dummy Data Generation (for demonstration only, REPLACE with your actual data loading) ---
def _generate_dummy_data(num_entries, start_time, market_type="spot"):
    data = []
    current_time = start_time
    base_price = 100.0 if market_type == "spot" else 100.1
    for i in range(num_entries):
        # Simulate more realistic tick frequency distribution with higher chance of smaller deltas
        time_delta_ms = np.random.choice([0, 1, 2, 3, 5, 10, 50, 100], p=[0.25, 0.25, 0.2, 0.1, 0.05, 0.05, 0.05, 0.05])
        current_time += timedelta(microseconds=time_delta_ms * 1000)
        
        price_change = np.random.uniform(-0.05, 0.05)
        current_price = base_price * (1 + price_change / 1000) # Smaller price changes
        
        spread = np.random.uniform(0.01, 0.05)
        best_bid = current_price - spread / 2
        best_ask = current_price + spread / 2
        
        trade_price = np.random.uniform(best_bid, best_ask)
        trade_quantity = np.random.randint(1, 100)
        
        entry = {
            'timestamp': current_time,
            'best_bid': best_bid,
            'best_ask': best_ask,
            'trade_price': trade_price if market_type == "spot" else np.nan,
            'trade_quantity': trade_quantity if market_type == "spot" else np.nan
        }
        data.append(entry)
        base_price = current_price
    df = pd.DataFrame(data)
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df = df.set_index('timestamp').sort_index()
    return df

# Create dummy data files for demonstration
# Using a larger number of entries to simulate real data size challenges
dummy_start_time = datetime(2023, 1, 1, 9, 0, 0)
_dummy_spot_df = _generate_dummy_data(100000, dummy_start_time, market_type="spot")
_dummy_perpetual_df = _generate_dummy_data(100000, dummy_start_time, market_type="perpetual")

# Save dummy data to CSV. In your real scenario, these files would already exist.
_dummy_spot_df.to_csv('dummy_spot_data.csv', index=True)
_dummy_perpetual_df.to_csv('dummy_perpetual_data.csv', index=True)
logger.info("Dummy data generated and saved to 'dummy_spot_data.csv' and 'dummy_perpetual_data.csv'.")
# --- END Dummy Data Generation ---

# Load your actual data here:
spot_data = load_your_spot_data('dummy_spot_data.csv') # REPLACE with your actual spot data path
perpetual_data = load_your_perpetual_data('dummy_perpetual_data.csv') # REPLACE with your actual perpetual data path

logger.info("\nInitial Data Load Complete.")
logger.info("Spot Data Head:\n%s", spot_data.head())
logger.info("\nPerpetual Data Head:\n%s", perpetual_data.head())

# --- 1. Data Pre-processing and Feature Engineering ---
logger.info("\n--- 1. Data Pre-processing and Feature Engineering ---")

# 1.2 Definition and Calculation of Mid-Price, Bid/Ask Spreads, and Other Microstructure Features
spot_data['mid_price'] = (spot_data['best_bid'] + spot_data['best_ask']) / 2
perpetual_data['mid_price'] = (perpetual_data['best_bid'] + perpetual_data['best_ask']) / 2

spot_data['bid_ask_spread'] = spot_data['best_ask'] - spot_data['best_bid']
perpetual_data['bid_ask_spread'] = perpetual_data['best_ask'] - perpetual_data['best_bid']

spot_data['best_bid_change'] = spot_data['best_bid'].diff()
spot_data['best_ask_change'] = spot_data['best_ask'].diff()
perpetual_data['best_bid_change'] = perpetual_data['best_bid'].diff()
perpetual_data['best_ask_change'] = perpetual_data['best_ask'].diff()

logger.info("\nFeatures added to Spot Data (head):\n%s", spot_data[['mid_price', 'bid_ask_spread', 'best_bid_change', 'best_ask_change']].head())

# 1.3 Leveraging Trade Data: Calculation and Predictive Power of Order Flow Imbalance (OFI)
def classify_trade_direction(trades_df, quotes_df):
    trades_df_sorted = trades_df.sort_index()
    quotes_df_sorted = quotes_df[['best_bid', 'best_ask']].sort_index()

    merged_df = pd.merge_asof(trades_df_sorted, quotes_df_sorted,
                              left_index=True, right_index=True, direction='backward')
    
    trade_direction = pd.Series(0, index=merged_df.index)
    trade_direction[merged_df['trade_price'] >= merged_df['best_ask']] = 1
    trade_direction[merged_df['trade_price'] <= merged_df['best_bid']] = -1
    
    return trade_direction

def calculate_trade_based_ofi(df):
    df_copy = df.copy()
    df_copy['trade_direction'] = classify_trade_direction(df_copy, df_copy) 
    df_copy['signed_trade_volume'] = df_copy['trade_quantity'] * df_copy['trade_direction']
    df_copy['OFI'] = df_copy['signed_trade_volume'].rolling('100ms', closed='left').sum().fillna(0)
    return df_copy

# Apply OFI calculation to spot data (as it has trade data)
spot_data = calculate_trade_based_ofi(spot_data)
logger.info("\nSpot Data with OFI (head):\n%s", spot_data[['mid_price', 'OFI']].head())


# --- 2. Quantifying Lead-Lag Relationships and Price Discovery ---
logger.info("\n--- 2. Quantifying Lead-Lag Relationships and Price Discovery ---")

spot_series = spot_data['mid_price']
perpetual_series = perpetual_data['mid_price']

try:
    # Use a smaller sample for lead_lag if your full 24h data is too large for it to complete in reasonable time
    # For robust analysis, you'd apply this to segmented windows of your full 24h data.
    sample_size_for_lag = min(len(spot_series), len(perpetual_series), 100000) # Limit for demonstration
    estimated_lag_spot_perpetual = lag(
        spot_series.iloc[:sample_size_for_lag],
        perpetual_series.iloc[:sample_size_for_lag],
        max_lag=0.01 # 10 milliseconds
    )

    logger.info(f"\nEstimated Lead-Lag (Spot vs Perpetual Mid-Price): {estimated_lag_spot_perpetual * 1000:.2f} ms")
    if estimated_lag_spot_perpetual is not None:
        if estimated_lag_spot_perpetual > 0:
            logger.info("Interpretation: Spot market leads the Perpetual market (positive lag means ts1 leads ts2).")
        elif estimated_lag_spot_perpetual < 0:
            logger.info("Interpretation: Perpetual market leads the Spot market (negative lag means ts2 leads ts1).")
        else:
            logger.info("Interpretation: No significant lead-lag detected within the max_lag window.")
    else:
        logger.info("Could not estimate lead-lag. Ensure 'lead_lag' library is installed and data is sufficient.")
except Exception as e:
    logger.error(f"Error calculating lead-lag: {e}")

avg_spot_spread = spot_data['bid_ask_spread'].mean()
avg_perpetual_spread = perpetual_data['bid_ask_spread'].mean()
avg_spot_trade_volume = spot_data['trade_quantity'].mean()

logger.info(f"\nAverage Spot Bid-Ask Spread: {avg_spot_spread:.4f}")
logger.info(f"Average Perpetual Bid-Ask Spread: {avg_perpetual_spread:.4f}")
logger.info(f"Average Spot Trade Volume: {avg_spot_trade_volume:.2f}")


# --- 3. Predictive Modeling for Cross-Market Trading Signals ---
logger.info("\n--- 3. Predictive Modeling for Cross-Market Trading Signals ---")

# 3.1 Defining and Detecting "Sudden Price Changes"
def detect_sudden_price_change(df, price_col, threshold_bps=7, time_window_ms=3):
    df_copy = df.copy()
    df_copy['price_prev'] = df_copy[price_col].shift(1)
    df_copy['time_prev'] = df_copy.index.to_series().shift(1)
    
    df_copy['pct_change'] = (df_copy[price_col] - df_copy['price_prev']) / df_copy['price_prev'] * 10000
    df_copy['time_diff_ms'] = (df_copy.index.to_series() - df_copy['time_prev']).dt.total_seconds() * 1000
    
    sudden_change_mask = (df_copy['time_diff_ms'] <= time_window_ms) & \
                         (df_copy['pct_change'].abs() >= threshold_bps)
    
    return sudden_change_mask.fillna(False)

spot_data['sudden_change'] = detect_sudden_price_change(spot_data, 'mid_price')
perpetual_data['sudden_change'] = detect_sudden_price_change(perpetual_data, 'mid_price')

logger.info(f"\nNumber of sudden price changes in Spot data: {spot_data['sudden_change'].sum()}")
logger.info(f"Number of sudden price changes in Perpetual data: {perpetual_data['sudden_change'].sum()}")


# 3.2 Proposed Machine Learning Models for Ultra-Short-Term Price Prediction
def prepare_data_for_ml(source_df, target_df, prediction_horizon_ms=5, change_threshold_bps=7):
    """
    Prepares features and labels for ML model, specifically designed for irregular time series.
    Features are from `source_df`. Labels are future price movements in `target_df`.
    This function avoids full resampling to prevent MemoryError.
    """
    logger.info("Preparing data for ML. This might take some time for large datasets.")

    # Select relevant features from the source DataFrame
    # Ensure these columns exist in your actual data.
    # We create a copy to avoid SettingWithCopyWarning.
    source_features = source_df[['mid_price', 'bid_ask_spread', 'best_bid_change', 'best_ask_change', 'OFI']].copy()
    
    # Drop rows with NaN from initial diff() calculations for features
    source_features.dropna(inplace=True)

    # Align current target prices with source_features' timestamps (backward merge_asof)
    # This gets the target price at or just before each source feature's timestamp
    aligned_current_target_price = pd.merge_asof(
        source_features.index.to_frame(name='timestamp'),
        target_df['mid_price'].to_frame(name='mid_price_target'),
        left_on='timestamp',
        right_index=True,
        direction='backward'
    ).set_index('timestamp')['mid_price_target']

    # Align future target prices with prediction horizon (forward merge_asof)
    # This gets the target price at or just after (timestamp + prediction_horizon)
    future_timestamps = source_features.index + timedelta(milliseconds=prediction_horizon_ms)
    aligned_future_target_price = pd.merge_asof(
        future_timestamps.to_frame(name='timestamp'),
        target_df['mid_price'].to_frame(name='mid_price_target'),
        left_on='timestamp',
        right_index=True,
        direction='forward', # Look forward for the price after the horizon
        tolerance=timedelta(milliseconds=prediction_horizon_ms) # Optional: only merge if within tolerance
    ).set_index('timestamp')['mid_price_target']

    # Combine current and future prices, aligning by the original source_features index
    combined_prices = pd.DataFrame({
        'current_target_price': aligned_current_target_price,
        'future_target_price': aligned_future_target_price
    }, index=source_features.index)

    # Drop any rows where we couldn't find both current and future prices
    combined_prices.dropna(inplace=True)

    # Calculate the percentage change for labels
    combined_prices['target_future_pct_change'] = (
        combined_prices['future_target_price'] - combined_prices['current_target_price']
    ) / combined_prices['current_target_price'] * 10000 # in bps

    # Define labels: 0 for no significant change, 1 for up, -1 for down
    labels = pd.Series(0, index=combined_prices.index)
    labels[combined_prices['target_future_pct_change'] >= change_threshold_bps] = 1 # Upward move
    labels[combined_prices['target_future_pct_change'] <= -change_threshold_bps] = -1 # Downward move
    
    # Align features (X) with the calculated labels (y) based on their common index
    X = source_features.loc[labels.index]
    y = labels

    logger.info(f"Finished preparing data for ML. Features shape: {X.shape}, Labels shape: {y.shape}")
    return X, y

# 3.3 Development and Evaluation of Trading Signals
def train_and_evaluate_model(X, y, model_name="SVM"):
    if X.empty or y.empty or len(y.unique()) < 2:
        logger.warning(f"Not enough data or classes to train for {model_name}. Skipping training.")
        return None, None
        
    # Scale features - essential for SVMs and many other ML models
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    X_scaled_df = pd.DataFrame(X_scaled, index=X.index, columns=X.columns) # Keep DataFrame structure

    # Stratify split to ensure representation of all classes
    X_train, X_test, y_train, y_test = train_test_split(X_scaled_df, y, test_size=0.3, random_state=42, stratify=y)
    
    if model_name == "SVM":
        model = SVC(kernel='rbf', random_state=42, probability=True)
    else:
        raise ValueError("Unsupported model type.")
        
    logger.info(f"\nTraining {model_name} model...")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    accuracy = accuracy_score(y_test, y_pred)
    
    from sklearn.metrics import precision_recall_fscore_support
    precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred, labels=[1, -1], average=None, zero_division=0)
    
    precision_up = precision[0] if 1 in model.classes_ else 0
    recall_up = recall[0] if 1 in model.classes_ else 0
    f1_up = f1[0] if 1 in model.classes_ else 0

    precision_down = precision[1] if -1 in model.classes_ else 0
    recall_down = recall[1] if -1 in model.classes_ else 0
    f1_down = f1[1] if -1 in model.classes_ else 0
    
    logger.info(f"--- {model_name} Performance ---")
    logger.info(f"Overall Accuracy: {accuracy:.4f}")
    logger.info(f"Precision (Up): {precision_up:.4f}, Recall (Up): {recall_up:.4f}, F1-Score (Up): {f1_up:.4f}")
    logger.info(f"Precision (Down): {precision_down:.4f}, Recall (Down): {recall_down:.4f}, F1-Score (Down): {f1_down:.4f}")
    
    return model, y_pred

# Train and evaluate models for both directions
try:
    logger.info("\n--- Training Spot to Perpetual Prediction Model ---")
    X_spot_to_perpetual, y_spot_to_perpetual = prepare_data_for_ml(spot_data, perpetual_data)
    svm_spot_to_perpetual, y_pred_spot_to_perpetual = train_and_evaluate_model(X_spot_to_perpetual, y_spot_to_perpetual)

    logger.info("\n--- Training Perpetual to Spot Prediction Model ---")
    X_perpetual_to_spot, y_perpetual_to_spot = prepare_data_for_ml(perpetual_data, spot_data)
    svm_perpetual_to_spot, y_pred_perpetual_to_spot = train_and_evaluate_model(X_perpetual_to_spot, y_perpetual_to_spot)

    logger.info(f"\nShape of features (Spot to Perpetual): {X_spot_to_perpetual.shape}")
    logger.info(f"Distribution of labels (Spot to Perpetual):\n{y_spot_to_perpetual.value_counts()}")

except Exception as e:
    logger.error(f"Error preparing data for ML or training models: {str(e)}")
    sys.exit(1)


# --- 4. Characterizing Noise and Momentum in High-Frequency Data ---
logger.info("\n--- 4. Characterizing Noise and Momentum in High-Frequency Data ---")

# 4.1 Identification and Statistical Characterization of Noise Events
def identify_noise_events(df, price_col, sudden_change_mask, reversion_window_ms=10, reversion_threshold_bps=5):
    noise_events = pd.Series(False, index=df.index)
    
    for idx in df[sudden_change_mask].index:
        pre_change_price_idx = df.index.get_loc(idx) - 1
        if pre_change_price_idx < 0:
            continue
        pre_change_price = df[price_col].iloc[pre_change_price_idx]
        
        reversion_end_time = idx + timedelta(milliseconds=reversion_window_ms)
        
        window_prices = df.loc[idx:reversion_end_time][price_col]
        
        if not window_prices.empty:
            reverted = (np.abs(window_prices - pre_change_price) / pre_change_price * 10000) <= reversion_threshold_bps
            if reverted.any():
                noise_events.loc[idx] = True
                
    return noise_events

spot_data['noise_event'] = identify_noise_events(spot_data, 'mid_price', spot_data['sudden_change'])
logger.info(f"\nNumber of noise events detected in Spot data: {spot_data['noise_event'].sum()}")

# 4.2 Techniques for Filtering Market Microstructure Noise
# Note: Median filter requires a regularly sampled series.
# If you apply this to your full 24h data, resampling to 1ms will be memory intensive.
# Consider applying this only to smaller, relevant time windows or for visualization,
# not necessarily for direct input to the ML model if the model prefers irregular data.
try:
    spot_mid_price_resampled = spot_data['mid_price'].resample('1ms').ffill()
    perpetual_mid_price_resampled = perpetual_data['mid_price'].resample('1ms').ffill()

    window_size = 5 # kernel_size must be odd
    if len(spot_mid_price_resampled) >= window_size:
        spot_data['mid_price_filtered'] = pd.Series(
            medfilt(spot_mid_price_resampled.values, kernel_size=window_size),
            index=spot_mid_price_resampled.index
        )
    else:
        spot_data['mid_price_filtered'] = spot_data['mid_price'] # No filtering if data too short

    if len(perpetual_mid_price_resampled) >= window_size:
        perpetual_data['mid_price_filtered'] = pd.Series(
            medfilt(perpetual_mid_price_resampled.values, kernel_size=window_size),
            index=perpetual_mid_price_resampled.index
        )
    else:
        perpetual_data['mid_price_filtered'] = perpetual_data['mid_price'] # No filtering if data too short

    logger.info(f"\nOriginal Spot Mid-Price (first 10, resampled): {spot_mid_price_resampled.iloc[:10].tolist()}")
    logger.info(f"Filtered Spot Mid-Price (first 10, resampled): {spot_data['mid_price_filtered'].iloc[:10].tolist()}")
except MemoryError as e:
    logger.error(f"MemoryError during noise filtering (resampling to 1ms): {e}. This step can be very memory intensive for long durations. Consider skipping or performing on smaller data subsets.")
except Exception as e:
    logger.error(f"Error during noise filtering: {e}")


# 4.3 Analysis of Momentum Quality and Persistence
def analyze_momentum_persistence(df, price_col, sudden_change_mask, momentum_window_ms=10, initial_change_threshold_bps=7):
    durations = []
    price_deltas = []

    for idx in df[sudden_change_mask].index:
        initial_price = df.loc[idx, price_col]
        
        prev_idx_loc = df.index.get_loc(idx) - 1
        if prev_idx_loc < 0:
            continue
        price_before_change = df[price_col].iloc[prev_idx_loc]
        
        price_diff_bps = (initial_price - price_before_change) / price_before_change * 10000
        
        initial_direction = 0
        if price_diff_bps >= initial_change_threshold_bps:
            initial_direction = 1
        elif price_diff_bps <= -initial_change_threshold_bps:
            initial_direction = -1
        else:
            continue
            
        momentum_end_time = idx + timedelta(milliseconds=momentum_window_ms)
        momentum_segment = df.loc[idx:momentum_end_time]
        
        if momentum_segment.empty:
            continue

        if initial_direction == 1:
            peak_price = momentum_segment[price_col].max()
            peak_time = momentum_segment[price_col].idxmax()
            additional_delta = (peak_price - initial_price) / initial_price * 10000
            if additional_delta > 0:
                durations.append((peak_time - idx).total_seconds() * 1000)
                price_deltas.append(additional_delta)
                
        elif initial_direction == -1:
            trough_price = momentum_segment[price_col].min()
            trough_time = momentum_segment[price_col].idxmin()
            additional_delta = (initial_price - trough_price) / initial_price * 10000
            if additional_delta > 0:
                durations.append((trough_time - idx).total_seconds() * 1000)
                price_deltas.append(additional_delta)
                
    return np.mean(durations) if durations else 0, np.mean(price_deltas) if price_deltas else 0

avg_momentum_duration_ms, avg_price_delta_persistence_bps = analyze_momentum_persistence(
    spot_data, 'mid_price', spot_data['sudden_change'], momentum_window_ms=10
)

logger.info(f"\nAverage Momentum Duration (Spot): {avg_momentum_duration_ms:.2f} ms")
logger.info(f"Average Price Delta Persistence (Spot): {avg_price_delta_persistence_bps:.2f} bps")

logger.info("\n--- End of Code Demonstration ---")


TypeError: unsupported type for timedelta microseconds component: numpy.int64

In [2]:
SPOT_PATH = '../data/trb_usdt_spot_export.csv'
PERP_PATH = '../data/trb_usdt_futures_export.csv'
TRADES_PATH = '../data/trb_usdt_trades_export.csv'

# Trying solution proposed